# FFCell — PART29 실행용 노트북 (투트랙: 실시간 알림 + What-if, n8n v7 필요)

이 파일은 **PART29(투트랙 알림)까지 딱 실행하기 위한** 최소 구성 노트북입니다. 실제로
n8n에 전송하는 셀은 맨 마지막 **딱 하나**뿐입니다 (중복 전송 없음).

- n8n은 반드시 **`ffcell_n8n_workflow_v7_whatif워크스페이스반영.json`**을 import(+Publish)한 뒤
  쓰세요.
- **실행 전 꼭 하실 것 (2026-07-29 최종 확인)**: What-if 메시지는 신규 Slack 워크스페이스
  (whatif-uof4216)의 **`whatif-투트랙-채널-전체`** 채널로 갑니다. n8n에 이 워크스페이스용
  **Slack Credential을 새로 등록**하고 `Slack 전송 (What-if)` 노드에 연결해주셔야 실제로
  전송됩니다.
- 2024년 원본 파일(R01_Data_2024.xlsx, R03_Data_2024.xlsx, FFCell_CycleManagement_2024.xlsx)이
  Drive에 있어야 합니다.
- 실행 방법: Runtime → Run all. 맨 마지막 셀(PART29-2)이 실시간 알림 + What-if를 함께 보냅니다.

# [분할 작업본 4/4] FFCell v80 (PART 번호 연속 정리판) — 튜터 피드백 반영 · 버그 수정 · 최종 파이프라인 표준화

- 범위: PART 15~19 (cell 396~456)
- 원본: `FFCell_integrated_notebook_v80_PART연속.ipynb` (PART 15~19 번호를 연속으로 재정리한 버전)

---


In [ ]:
!pip install scikit-posthocs imblearn --quiet

In [ ]:
# ── [분할 파일 시작] 이전 파일(part14) 세션 복원 ─────────────────────────────
# 이 파일은 FFCell_integrated_notebook_v80(PART 번호 연속 정리판)을 작업용으로
# 4등분한 것 중 "최종 마무리" (PART 15~19) 부분이다. 아래 셀로 이전 파일 끝 시점
# 전체 상태를(함수·변수·df 전부) 그대로 복원한 뒤, 이어지는 셀부터 실행하면 된다.
from pathlib import Path
_CACHE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/FFCell/_checkpoints')

from google.colab import drive
drive.mount('/content/drive')

try:
    import dill
except ImportError:
    !pip install dill --quiet
    import dill

_session_path = _CACHE_DIR / 'part14_session.pkl'
if _session_path.exists():
    dill.load_session(str(_session_path))
    print(f"[세션 복원 완료] {_session_path} ← 이전 파일 상태를 그대로 이어받았습니다.")
else:
    print(f"[체크포인트 없음] {_session_path} 파일이 없습니다.")
    print("  → 이전 분할 파일을 먼저 끝까지 실행해서 체크포인트를 만들어두세요.")
    raise FileNotFoundError(
        f"세션 체크포인트({_session_path})가 없어 이전 PART(1~14)의 변수·데이터프레임을 "
        "복원할 수 없습니다. 아래 셀부터는 이 상태에 의존하므로, 여기서 실행을 멈추고 "
        "이전 분할 파일(part14)을 먼저 끝까지 실행해 체크포인트를 만든 뒤 다시 시도하세요."
    )


Mounted at /content/drive
[세션 복원 완료] /content/drive/MyDrive/Colab Notebooks/FFCell/_checkpoints/part14_session.pkl ← 이전 파일 상태를 그대로 이어받았습니다.


### PART21-0. Google Drive 마운트 + 파일 자동 검색

공유 드라이브(`공유 문서함 > 이미지 데이터 > 24년 csv`)에 있는 파일이라 정확한 절대경로를
외우기 번거로우니, 마운트 후 **파일명으로 자동 검색**해서 경로를 찾습니다.

In [ ]:
# ── Google Drive 마운트 ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ── 파일명으로 자동 검색 (공유 드라이브 포함 전체 검색) ────────────────
import subprocess

def find_file(filename, search_root='/content/drive'):
    result = subprocess.run(
        ['find', search_root, '-iname', filename],
        capture_output=True, text=True
    )
    paths = [p for p in result.stdout.strip().split('\n') if p]
    if not paths:
        raise FileNotFoundError(f'{filename} 을(를) {search_root} 아래에서 찾지 못했습니다. '
                                 f'해당 공유 드라이브가 "내 드라이브에 바로가기 추가"로 연결되어 있는지 확인하세요.')
    if len(paths) > 1:
        print(f'⚠️ {filename} 이(가) {len(paths)}개 발견됨 — 첫 번째 경로 사용:')
        for p in paths:
            print('  -', p)
    return paths[0]

R03_PATH_FOUND = find_file('R03_Data_2024.xlsx')
CYCLE_MANAGEMENT_PATH_FOUND = find_file('FFCell_CycleManagement_2024.xlsx')
print('R03_Data_2024.xlsx        ->', R03_PATH_FOUND)
print('FFCell_CycleManagement_2024.xlsx ->', CYCLE_MANAGEMENT_PATH_FOUND)

R03_Data_2024.xlsx        -> /content/drive/MyDrive/R03_Data_2024.xlsx
FFCell_CycleManagement_2024.xlsx -> /content/drive/MyDrive/FFCell_CycleManagement_2024.xlsx


In [ ]:
# ── PART21-1. 설정 ─────────────────────────────────────────────
import os
import time
import json as _json
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import requests

# n8n 웹훅 URL — 본인 환경에 맞게 교체 (ffcell_n8n_workflow.json import 후 웹훅 노드 URL 복사)
N8N_WEBHOOK_URL = "YOUR_N8N_WEBHOOK_URL"

# 재생 간격(초) — 사이클마다 이만큼 쉬고 다음 사이클을 보낸다 (데모 속도 조절용)
REPLAY_INTERVAL_SEC = 1.5

# 24년 원본 데이터 경로 (Colab 작업 경로 기준, 필요시 Drive 마운트 경로로 수정)
R03_PATH_2024 = R03_PATH_FOUND
CYCLE_MANAGEMENT_PATH_2024 = CYCLE_MANAGEMENT_PATH_FOUND

R03_COL = "I_R03_Gripper_Load"

# 23년 데이터로 이미 확정된 고정 임계값 (재계산 금지 — 그래야 "일반화 테스트"가 성립함)
NOSE_THRESH_FIXED = 3110.5
BODY2_THRESH_FIXED = 4961.0
BODY1_THRESH_FIXED = 5173.5   # 23년 원본 R01~R04 데이터로 재구성 확정 (본 노트북 별도 검증)

# 판정 시간창 (PART20 확정값, 사이클 시작 기준 초)
BODY1_WINDOW = (62.8015, 68.8015)   # 65.8015 ± 3
BODY2_WINDOW = (75.807, 81.807)     # 78.807  ± 3
NOSE_WINDOW = (93.0, 101.0)         # 슬롯3 추정 중심 ± 4

In [ ]:
# ── (2026-08-04 업데이트) 대시보드 대표 이미지 사용 — zip 대신 Drive의 대표 이미지 직접 사용 ──
import base64

# 결함 유형 -> 새로 만든 Drive 폴더(FFCell_n8n_demo_images)에 올려둔 대시보드 대표 이미지 파일명
# find_file()이 파일명으로 Drive 전체를 재귀 검색하므로 폴더 위치는 어디든 상관없다.
#
# Normal은 의도적으로 포함하지 않음 -- 정상은 확인할 이상이 없어 이미지의 효용이 낮고,
# "이미지가 붙었다 = 확인이 필요하다"는 신호를 명확하게 유지하기 위해 결함일 때만 첨부한다
# (아래 build_cycle_complete_payload의 is_defect 체크와 짝을 이루는 설계).
DASHBOARD_IMAGE_MAP = {
    "NoNose": "sample_nonose.png",
    "NoNose,NoBody2": "sample_nonose_nobody2.png",
    "NoNose,NoBody2,NoBody1": "sample_triple.png",
}

IMAGE_BASE64_CACHE = {}
for label, filename in DASHBOARD_IMAGE_MAP.items():
    path = find_file(filename)
    with open(path, "rb") as f:
        data = f.read()
    IMAGE_BASE64_CACHE[label] = (base64.b64encode(data).decode("utf-8"), filename)
    print(f"{label} 대표 이미지: {path}")


NoNose 대표 이미지: /content/drive/MyDrive/FFCell_n8n_demo_images/sample_nonose.png
NoNose,NoBody2 대표 이미지: /content/drive/MyDrive/FFCell_n8n_demo_images/sample_nonose_nobody2.png
NoNose,NoBody2,NoBody1 대표 이미지: /content/drive/MyDrive/FFCell_n8n_demo_images/sample_triple.png


In [ ]:
def encode_image_base64(label):
    """캐싱해둔 대표 이미지의 (base64문자열, 파일명) 반환.
    Normal은 DASHBOARD_IMAGE_MAP에 없으므로 (None, None)이 반환되고,
    build_cycle_complete_payload의 is_defect 체크에 의해 애초에 호출되지 않는다."""
    return IMAGE_BASE64_CACHE.get(label, (None, None))


In [ ]:
# ── PART21-2(보강). 누락된 정의 복원 ──────────────────────────────────
DEFECT_KOR = {
    "Normal": "정상",
    "NoNose": "노즈 미조립",
    "NoNose,NoBody2": "노즈+바디2 미조립",
    "NoNose,NoBody2,NoBody1": "3중 결손",
}


def rule_based_predict_replay(row):
    """PART20 원본(v80_3of4 71장 rule_final_from_flags / 48장 rule_based_predict)과
    동일한 계층 순서로 복원 — Nose -> Body2 -> Body1 순으로 '존재(>= 임계값)' 확인.
    (2026-07-28: v80_3of4 원본 파일 대조로 검증 완료 — 이전에 Body1 우선으로 잘못
    재구성했던 버전을 이 순서로 교체함)"""
    nose_present = pd.notna(row["nose_force"]) and row["nose_force"] >= NOSE_THRESH_FIXED
    if nose_present:
        return "Normal"
    body2_present = pd.notna(row["body2_force"]) and row["body2_force"] >= BODY2_THRESH_FIXED
    if body2_present:
        return "NoNose"
    body1_present = pd.notna(row["body1_force"]) and row["body1_force"] >= BODY1_THRESH_FIXED
    if body1_present:
        return "NoNose,NoBody2"
    return "NoNose,NoBody2,NoBody1"


def send_to_n8n(payload):
    """n8n 웹훅으로 payload 전송. n8n은 판정을 다시 하지 않고 감시·전달만 한다는
    원칙에 따라, 이미 계산된 결과를 그대로 전달한다."""
    try:
        resp = requests.post(N8N_WEBHOOK_URL, json=payload, timeout=10)
        print(f"  -> n8n 응답: {resp.status_code}")
        return resp
    except Exception as e:
        print(f"  [WARN] n8n 전송 실패: {e}")
        return None


def build_summary_payload(feat, note=""):
    """전체 사이클 재생 종료 후 보내는 요약 payload.
    (2026-07-28: 실제 ffcell_n8n_workflow.json의 '요약 메시지 조립' 코드가 기대하는
    필드명(defect_counts_kor, accuracy_vs_actual_label, note)에 맞춰 재구성함 --
    이전 버전은 accuracy/macro_recall/defect_cycles라는 다른 이름을 써서 n8n에서
    undefined로 표시됐을 것)"""
    from sklearn.metrics import accuracy_score
    y_true = feat["label"]
    y_pred = feat["pred"]
    acc = accuracy_score(y_true, y_pred)

    # n8n 쪽 Slack 메시지가 "결함유형별 건수"를 그대로 나열하므로, 한글 라벨 기준으로 집계
    defect_counts_kor = {}
    for label, kor in DEFECT_KOR.items():
        if label == "Normal":
            continue
        cnt = int((feat["pred"] == label).sum())
        if cnt > 0:
            defect_counts_kor[kor] = cnt

    return {
        "type": "summary",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "total_cycles": int(len(feat)),
        "defect_counts_kor": defect_counts_kor,
        "accuracy_vs_actual_label": round(float(acc), 4),
        "note": note,
    }

In [ ]:
# ── PART24-2. 단계 정의 및 판정 헬퍼 ─────────────────────────────────────
STAGES = [
    ("body1", BODY1_WINDOW, BODY1_THRESH_FIXED, "Body1"),
    ("body2", BODY2_WINDOW, BODY2_THRESH_FIXED, "Body2"),
    ("nose",  NOSE_WINDOW,  NOSE_THRESH_FIXED,  "Nose"),
]

COMPRESSION_FACTOR = 20  # 실제 초 -> 데모 대기 초 압축 배율 (필요시 조정)


def stage_peak(g, window, col=R03_COL):
    a, b = window
    sub = g.loc[g["elapsed_s"].between(a, b), col].dropna()
    return sub.max() if not sub.empty else np.nan


def finalize_label_from_presence(present):
    """PART21-2보강에서 검증한 것과 동일한 우선순위(Nose->Body2->Body1)로 최종 라벨 확정.
    단계별 조기경보와 별개로, 통계·정확도 계산용 최종 판정은 이 함수로 통일한다."""
    if present.get("nose"):
        return "Normal"
    if present.get("body2"):
        return "NoNose"
    if present.get("body1"):
        return "NoNose,NoBody2"
    return "NoNose,NoBody2,NoBody1"


def build_stage_alert_payload(cycle_id, stage_kor, window_close_s, peak_value, threshold):
    return {
        "type": "stage_alert",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "cycle_id": int(cycle_id),
        "stage": stage_kor,
        "window_close_s": window_close_s,
        "peak_value": None if pd.isna(peak_value) else round(float(peak_value), 1),
        "threshold": threshold,
        "message": f"[조기경보] 사이클 {cycle_id} — {stage_kor} 단계에서 신호 미확인 "
                    f"(사이클 완료 대기 없이 {window_close_s}초 시점에 즉시 발신)",
    }


def build_cycle_complete_payload(cycle_id, label, final_pred, already_alerted):
    is_defect = final_pred != "Normal"
    payload = {
        "type": "cycle_complete",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "cycle_id": int(cycle_id),
        "final_pred": final_pred,
        "final_pred_kor": DEFECT_KOR.get(final_pred, final_pred),
        "actual_label": label,
        "correct": bool(final_pred == label),
        "already_alerted_early": already_alerted,
        "is_defect": bool(is_defect),
    }
    # PART21과 동일 -- 최종 확정 시점(cycle_complete)에 결함이면 대표 이미지 첨부
    if is_defect:
        img_b64, img_filename = encode_image_base64(final_pred)
        payload["image_base64"] = img_b64
        payload["image_filename"] = img_filename
    else:
        payload["image_base64"] = None
        payload["image_filename"] = None
    return payload

In [ ]:
# ── PART24-2(v2). R01 투입완료 '이벤트' 감지 + 상대시간 기준 단계 정의 ────────
def r01_settle_time_v2(g, r01_col="M_R01_BJointAngle_Degree"):
    """PART23에서 검증한 방식과 동일 -- 0~50초 구간 B축 각도 피크 이후,
    -88도 이하(정지 자세)로 돌아오는 첫 시점을 '투입완료 이벤트'로 감지한다."""
    window = g[g["elapsed_s"].between(0, 50)]
    if window.empty or window[r01_col].isna().all():
        return np.nan
    peak_idx = window[r01_col].idxmax()
    peak_time = g.loc[peak_idx, "elapsed_s"]
    after = g[g["elapsed_s"] > peak_time]
    settle = after[after[r01_col] <= -88]
    return settle["elapsed_s"].iloc[0] if len(settle) > 0 else np.nan


# PART22(Q1)에서 320개 사이클 기준으로 이미 계산해둔 투입완료 후 오프셋(대시보드 2p 반영값)
BODY1_OFFSET_FROM_SETTLE = 58.0
BODY2_OFFSET_FROM_SETTLE = 71.0
NOSE_OFFSET_FROM_SETTLE = 85.0

STAGES_REL = [
    ("body1", BODY1_OFFSET_FROM_SETTLE, 3.0, BODY1_THRESH_FIXED, "Body1"),
    ("body2", BODY2_OFFSET_FROM_SETTLE, 3.0, BODY2_THRESH_FIXED, "Body2"),
    ("nose",  NOSE_OFFSET_FROM_SETTLE,  4.0, NOSE_THRESH_FIXED,  "Nose"),
]
# v1 고정창(사이클 시작 기준) -- R01 이벤트를 못 잡았을 때만 폴백용으로 남겨둠
STAGES_FALLBACK_WINDOW = {"body1": BODY1_WINDOW, "body2": BODY2_WINDOW, "nose": NOSE_WINDOW}


def stage_peak_relative(g, settle_time, offset, margin, col=R03_COL):
    a, b = settle_time + offset - margin, settle_time + offset + margin
    sub = g.loc[g["elapsed_s"].between(a, b), col].dropna()
    return sub.max() if not sub.empty else np.nan

In [ ]:
# R01_Data_2024.xlsx 경로 찾기
R01_PATH_2024 = find_file('R01_Data_2024.xlsx')
print('R01_Data_2024.xlsx ->', R01_PATH_2024)

R01_Data_2024.xlsx -> /content/drive/MyDrive/R01_Data_2024.xlsx


In [ ]:
import bisect  # 사이클 경계 탐지에 필요

In [ ]:
# ── PART25-1. 폴링 기반 이벤트 감지 + 후속검증(verification flip) 추적 ────
POLL_INTERVAL_SEC_REAL = 1.5   # 멘토 피드백: 1~2초 대역 폴링
POLL_MAX_T_CAP = 120.0  # 2026-07-28 추가: 폴링 루프가 도는 '실제 초' 상한 (Nose 확인 시점 89초 + 여유)
                         # -- PART27처럼 사이클 구간이 다음 사이클 시작 직전까지 넓게 잡혀도,
                         #    3단계 확인이 다 끝난 뒤 남은 구간까지 쓸데없이 폴링하지 않도록 막는다.

FALLBACK_SETTLE_TIME = 15.41  # PART22에서 계산한 320개 사이클 중앙값 (이벤트 감지 실패 시 최후 대안)


def stream_cycle_v3(cycle_id, g, label, known_settle_time_s=None, verbose=True):
    # 2026-07-28 추가: PART27처럼 g의 elapsed_s=0이 '이미 자체 감지로 확정된 투입완료 시점'인
    # 경우, known_settle_time_s=0.0을 넘기면 이 값을 그대로 신뢰하고 r01_settle_time_v2로
    # 다시 찾으려 하지 않는다. (다시 찾으면 사이클 내 다른 소소한 R01 동작을 오탐할 수 있다 --
    # 2024 데이터 검증 중 발견된 버그: 8.8초 지점을 엉뚱하게 '투입완료'로 재감지하던 문제)
    settle_time = known_settle_time_s if known_settle_time_s is not None else np.nan
    settle_detected = known_settle_time_s is not None
    settle_fallback_used = False
    found_signal = {"body1": False, "body2": False, "nose": False}
    alert_fired = {}

    stages_order = [
        ("body1", BODY1_OFFSET_FROM_SETTLE, 3.0, BODY1_THRESH_FIXED, "Body1"),
        ("body2", BODY2_OFFSET_FROM_SETTLE, 3.0, BODY2_THRESH_FIXED, "Body2"),
        ("nose",  NOSE_OFFSET_FROM_SETTLE,  4.0, NOSE_THRESH_FIXED,  "Nose"),
    ]

    max_t = min(g["elapsed_s"].max(), POLL_MAX_T_CAP)
    t = 0.0

    while t <= max_t:
        window = g[g["elapsed_s"] <= t]

        # 1) R01 투입완료 이벤트 -- 매 폴링마다 재확인 (한 번 계산이 아니라 계속 지켜봄)
        if not settle_detected and len(window) > 0:
            settle_time_est = r01_settle_time_v2(window)
            if pd.notna(settle_time_est):
                settle_time = settle_time_est
                settle_detected = True
                if verbose:
                    print(f"  [폴링 감지] t={t:.1f}s -- R01 투입완료 이벤트 확인 (실제 {settle_time:.2f}s)")
            elif t >= 50:
                # 2026-07-28 버그 수정: 0~50초 구간에서 이벤트를 못 찾으면 그 이후로도
                # 절대 못 찾음(r01_settle_time_v2가 0~50초만 봄) -- 그대로 두면 세 단계를
                # 영원히 확인 안 해서 무조건 '3중결손'으로 굳어버린다. 중앙값으로 폴백해서
                # 늦게라도 확인은 계속 진행한다.
                settle_time = FALLBACK_SETTLE_TIME
                settle_detected = True
                settle_fallback_used = True
                if verbose:
                    print(f"  [경고] t={t:.1f}s -- R01 이벤트 감지 실패, 중앙값({FALLBACK_SETTLE_TIME}s)으로 폴백")

        # 2) 각 단계 폴링 확인 -- 신호를 찾을 때까지 사이클 끝까지 계속 확인
        if settle_detected:
            for key, offset, margin, thresh, kor in stages_order:
                if found_signal[key]:
                    continue
                peak = stage_peak_relative(window, settle_time, offset, margin)
                if pd.notna(peak) and peak >= thresh:
                    found_signal[key] = True
                    if verbose:
                        flip_note = " (조기경보 이후 뒤집힘!)" if key in alert_fired else ""
                        print(f"  [폴링 감지] t={t:.1f}s -- {kor} 신호 확인됨{flip_note}")
                    continue

                expected_close = settle_time + offset + margin
                if t >= expected_close and key not in alert_fired:
                    payload = build_stage_alert_payload(cycle_id, kor, round(t, 1), peak, thresh)
                    if verbose:
                        print(f"  {payload['message']}")
                    send_to_n8n(payload)
                    alert_fired[key] = t

        # 2026-07-28 버그 수정: t는 '압축 안 된 실제 초' 단위로 늘려야 59초·72초·85초 같은
        # 기준과 올바르게 비교된다. 대기(time.sleep)만 압축 배율만큼 짧게 재운다.
        t += POLL_INTERVAL_SEC_REAL
        time.sleep(POLL_INTERVAL_SEC_REAL / COMPRESSION_FACTOR)

    # 3) 최종 확정 -- 신호가 사이클 끝까지 안 나타난 단계는 '없음'으로 확정
    final_pred = finalize_label_from_presence(found_signal)
    already_alerted = len(alert_fired) > 0
    verification_flips = [k for k in alert_fired if found_signal[k]]  # 조기경보 후 뒤집힌 단계들

    complete_payload = build_cycle_complete_payload(cycle_id, label, final_pred, already_alerted)
    complete_payload["verification_flips"] = verification_flips
    complete_payload["settle_fallback_used"] = settle_fallback_used
    send_to_n8n(complete_payload)
    return complete_payload

In [ ]:
# ── build_line_stop_alert_payload (PART26-4에서 함수 정의만 재사용) ──
def build_line_stop_alert_payload(settle_time, idle_before_s, normal_median_s):
    return {
        "type": "line_stop_alert",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event_time": str(settle_time),
        "idle_before_s": round(float(idle_before_s), 1),
        "normal_median_s": round(float(normal_median_s), 1),
        "message": (
            f"[라인정지 의심] {settle_time} 직전 {idle_before_s:.0f}초 동안 R01 움직임이 없었습니다 "
            f"(정상 중앙값 {normal_median_s:.0f}초 대비 {idle_before_s/normal_median_s:.1f}배) -- "
            f"불량이 아니라 라인 자체가 멈췄을 가능성이 있습니다."
        ),
    }

## 143. PART27. 2024 데이터에 사이클 간격까지 반영한 실전형 통합 재생

**🟢 쉬운 설명**
지금까지(PART24·25)는 2024년 데이터를 재생할 때도 "지금이 몇 번째 사이클인지"를 `CycleManagement`
정답지가 미리 알려줬고, 한 사이클이 끝나면 **곧바로** 다음 사이클로 넘어갔어요. 실제로 이 시스템을
2024년 데이터(=실전 배포용) 위에서 돌린다면, 사이클과 사이클 사이에 실제로 쉬는 시간이 있고, 그
쉬는 시간이 끝나야 다음 사이클이 시작돼요. PART27은 PART26에서 검증한 "R01 신호만으로 사이클
시작을 스스로 알아채는 방법"과, PART25의 "1.5초 폴링으로 Body1·Body2·Nose를 확인하는 방법"을
합쳐서, **사이클 경계도 스스로 찾고, 사이클 사이 쉬는 시간도 실제로 흘려보내면서** 2024 데이터를
재생합니다. 즉 "정답지 없이, 진짜 라인처럼" 돌아가는 버전이에요.

**🔶 자세한 설명**
- `CycleManagement_2024`는 이제 사이클 경계를 알려주는 용도가 아니라, **자체 감지 결과가
  맞았는지 확인하는 정답지(검증용)**로만 씁니다 -- 감지 로직 자체는 R01+R03 원본 신호만 봅니다.
- 사이클 사이 쉬는 시간(`idle_before_s`)만큼 `COMPRESSION_FACTOR`로 압축해서 실제로
  `time.sleep()`합니다 -- 지금까지는 이 대기 자체가 아예 없었습니다.
- 쉬는 시간이 비정상적으로 길면(PART26에서 만든 라인정지 임계값 로직을 2024 데이터 기준으로
  다시 계산) `line_stop_alert`를 그대로 발신합니다.
- ⚠️ **주의**: 이 환경(샌드박스)에는 2024년 원본 파일(R01_Data_2024.xlsx 등)이 없어서, 아래
  코드는 실제로 실행해서 숫자를 검증하지는 못했습니다. Colab에서 실행하신 뒤 감지된 사이클
  수(93개에 가까운지), 쉬는 시간 분포가 이상하지 않은지 꼭 확인해주세요.

### PART27-1. 2024 데이터를 '사전 분할 없이' 연속 스트림으로 로드

In [ ]:
# ── PART27-1. CycleManagement는 검증용으로만 남기고, 실제 재생/감지는 R01+R03 연속 신호로 ──
def load_2024_continuous_stream():
    """PART24-1(v2)의 load_2024_raw_grouped_with_r01()과 달리, 사이클 단위로 미리 잘라놓지 않는다.
    감지 로직이 CycleManagement를 참조하지 못하도록 별도 변수로만 들고 있는다."""
    cyc_truth = pd.read_excel(CYCLE_MANAGEMENT_PATH_2024)   # 검증(정답 비교)용
    r03 = pd.read_excel(R03_PATH_2024)
    r01 = pd.read_excel(R01_PATH_2024)

    stream = r03.merge(r01[["_time", "M_R01_BJointAngle_Degree"]], on="_time", how="inner")
    stream["_time"] = pd.to_datetime(stream["_time"], format="ISO8601")
    stream = stream.sort_values("_time").reset_index(drop=True)

    cyc_truth["_time"] = pd.to_datetime(cyc_truth["_time"], format="ISO8601")
    cyc_truth = cyc_truth.sort_values("_time").reset_index(drop=True)
    return stream, cyc_truth


stream_2024, cyc_2024_truth = load_2024_continuous_stream()
print(f"연속 스트림 총 {len(stream_2024)}행 -- {stream_2024['_time'].min()} ~ {stream_2024['_time'].max()}")

연속 스트림 총 287954행 -- 2024-08-13 14:00:00.035000+00:00 ~ 2024-08-13 21:59:59.925000+00:00


### PART27-2. PART26과 동일한 방식으로 사이클 경계 자체 감지 (함수화)

In [ ]:
# ── PART27-2. R01 정지<->움직임 전환으로 사이클 시작 후보 찾기 (PART26 로직 재사용) ──
def detect_cycle_boundaries(stream_df, rest_thresh=-88.0, real_start_threshold=20.0):
    tz = stream_df["_time"].dt.tz  # 2026-07-28 버그 수정: .values로 뽑으면 타임존이 사라지므로 미리 기억
    angle = stream_df["M_R01_BJointAngle_Degree"].values
    times = stream_df["_time"].values
    resting = (angle <= rest_thresh).astype(int)
    diff = np.diff(resting)
    falling_idx = np.where(diff == -1)[0] + 1   # 움직이기 시작
    rising_idx = np.where(diff == 1)[0] + 1     # 정지 복귀(투입완료)
    rising_list = list(times[rising_idx])

    records = []
    for f_idx in falling_idx:
        ft = times[f_idx]
        pos_after = bisect.bisect_right(rising_list, ft)
        if pos_after >= len(rising_list):
            continue
        settle_time = rising_list[pos_after]
        pos_before = bisect.bisect_left(rising_list, ft)
        if pos_before == 0:
            continue  # 스트림 맨 처음이라 '직전' 정보가 없음 -- 첫 사이클은 판단 보류
        idle_before_s = (ft - rising_list[pos_before - 1]) / np.timedelta64(1, "s")
        records.append((ft, settle_time, idle_before_s))

    moves = pd.DataFrame(records, columns=["move_start", "settle_time", "idle_before_s"])
    if tz is not None and len(moves) > 0:
        # .values를 거치며 사라진 타임존을 원래대로 복원 (실제 시각은 그대로, 표시만 다시 붙임)
        moves["move_start"] = pd.to_datetime(moves["move_start"]).dt.tz_localize(tz)
        moves["settle_time"] = pd.to_datetime(moves["settle_time"]).dt.tz_localize(tz)

    real_starts = (
        moves[moves["idle_before_s"] >= real_start_threshold]
        .sort_values("settle_time")
        .drop_duplicates(subset="settle_time")
        .reset_index(drop=True)
    )
    return real_starts


detected_2024 = detect_cycle_boundaries(stream_2024)
print(f"2024 데이터에서 자체 감지한 사이클 시작: {len(detected_2024)}개 (실제 알려진 사이클 수: 93개)")
print(detected_2024["idle_before_s"].describe())

2024 데이터에서 자체 감지한 사이클 시작: 92개 (실제 알려진 사이클 수: 93개)
count     92.000000
mean     290.220511
std       35.709476
min      279.383000
25%      280.329750
50%      280.796000
75%      281.850500
max      475.692000
Name: idle_before_s, dtype: float64


### PART27-3. 정상 쉬는시간 범위 + 라인정지 임계값을 2024 데이터 기준으로 재산출

In [ ]:
# ── PART27-3. 임계값을 2023 데이터에서 고정값으로 가져오지 않고, 2024 데이터로 자동 산출 ──
gaps_2024 = detected_2024["idle_before_s"]
median_2024 = gaps_2024.median()
mad_2024 = (gaps_2024 - median_2024).abs().median()
LINE_STOP_THRESHOLD_2024 = median_2024 + 6 * mad_2024  # MAD 기반 -- 정규분포 가정 없이도 이상치에 안전

print(f"2024 정상 쉬는시간 중앙값: {median_2024:.1f}초, MAD: {mad_2024:.1f}초")
print(f"자동 산출된 라인정지 임계값: {LINE_STOP_THRESHOLD_2024:.1f}초")

long_gaps_2024 = detected_2024[gaps_2024 >= LINE_STOP_THRESHOLD_2024]
print(f"라인정지 의심 후보: {len(long_gaps_2024)}건")
if len(long_gaps_2024) == 0:
    print("(2024 데이터는 하루치 단일 세션이라, 세션 경계가 없으면 0건이 정상적으로 나올 수 있습니다)")

2024 정상 쉬는시간 중앙값: 280.8초, MAD: 0.6초
자동 산출된 라인정지 임계값: 284.1초
라인정지 의심 후보: 15건


In [ ]:
# ── lookup_actual_label_2024 (검증용) + 결함 포함 표본 구성 ──────────────
def lookup_actual_label_2024(settle_time, cyc_truth):
    """검증용으로만 사용 -- 판정 로직 자체는 이 함수를 참조하지 않는다.
    settle_time 시점에 실제로 활성화된 공식 사이클 번호를 먼저 찾고, 그 사이클
    전체에서 라벨을 조회한다 (사이클 경계 근처에서 옆 사이클 라벨을 잘못 가져오는 문제 방지)."""
    pos = cyc_truth["_time"].searchsorted(settle_time, side="right") - 1
    if pos < 0:
        return "Normal"
    cycle_num = cyc_truth["Q_Cell_CycleCount"].iloc[pos]
    match = cyc_truth[cyc_truth["Q_Cell_CycleCount"] == cycle_num]
    if "Description" not in match:
        return "Normal"
    vals = match["Description"].dropna().unique()
    return vals[0] if len(vals) > 0 else "Normal"


preview_labels = [
    lookup_actual_label_2024(detected_2024["settle_time"].iloc[i], cyc_2024_truth)
    for i in range(len(detected_2024))
]
normal_positions = [i for i, l in enumerate(preview_labels) if l == "Normal"]
defect_positions = [i for i, l in enumerate(preview_labels) if l != "Normal"]

# (2026-08-04 업데이트) "결함 중 처음 2개"를 순서대로 가져오던 기존 방식은 어떤 결함
# 유형이 앞쪽에 몰려있으면 그 유형만 반복해서 뽑힐 수 있다 (예: NoNose만 계속 걸림).
# 발표 데모에서 4종류(Normal/NoNose/NoNose,NoBody2/3중결손)를 전부 보여줘야 하므로,
# 라벨별로 정확히 1개씩 명시적으로 골라온다.
target_labels = ["Normal", "NoNose", "NoNose,NoBody2", "NoNose,NoBody2,NoBody1"]
sample_indices = []
for lbl in target_labels:
    idx = next((i for i, l in enumerate(preview_labels) if l == lbl), None)
    if idx is not None:
        sample_indices.append(idx)
    else:
        print(f"\u26a0\ufe0f {lbl} 라벨을 가진 사이클을 찾지 못했습니다 -- 2024 데이터 범위를 확인하세요.")

sample_indices = sorted(set(sample_indices))
n_defect_in_sample = len(set(sample_indices) & set(defect_positions))
print(f"표본 구성: {len(sample_indices)}개 (인덱스 {sample_indices}) -- 그중 결함 {n_defect_in_sample}개 포함")
print(f"라벨: {[preview_labels[i] for i in sample_indices]}")


표본 구성: 4개 (인덱스 [0, 24, 41, 59]) -- 그중 결함 3개 포함
라벨: ['Normal', 'NoNose', 'NoNose,NoBody2', 'NoNose,NoBody2,NoBody1']


## 145. PART28. 2024 데이터로 자체감지 정확도 재검증

**🟢 쉬운 설명**
PART26에서 "R01 신호만으로 사이클 경계를 스스로 찾는" 방법을 2023년 데이터(6개 세션)로
검증했었는데, 튜터님 피드백("원인을 파악해볼 것") 이후 다시 확인해보니, 2023년 초반 세션
3개(25개 사이클)는 **R01 각도 센서 값이 처음부터 끝까지 0으로만 찍혀 있어서** 애초에 감지가
불가능한 구간이었습니다. 이걸 빼고 나면 실제 감지율은 93.9%(283/302)였습니다. 그런데 실제로
n8n 데모에 쓰는 건 2024년 데이터인데, 정작 2024년 데이터로는 이 검증을 안 해봤다는 걸 최근
대화에서 짚게 되어, 이번에 2024년 데이터로도 똑같이 검증해봤습니다.

**🔶 자세한 설명**
2024년 데이터는 6개 세션으로 쪼개진 2023년과 달리 **하루치 연속 데이터(세션 1개)**라서,
R01 센서가 꺼져 있던 구간 같은 문제가 없습니다. 그 결과가 아래에 나옵니다.

In [ ]:
# ── PART28-1. 2024 데이터 자체감지 정확도 정밀 검증 (공식 사이클과 1:1 대조) ──
official_2024 = cyc_2024_truth.groupby("Q_Cell_CycleCount")["_time"].min().sort_values()
print(f"공식 전체 사이클 수(2024): {len(official_2024)}개")
print(f"자체 감지한 사이클 수: {len(detected_2024)}개")

detected_times_list = list(detected_2024["settle_time"])
TOL = 10.0  # 공식 시작과 자체감지 settle 사이의 자연적 시차(약 6~7초)를 감안한 허용오차

missed = []
for cnt, ot in official_2024.items():
    pos = bisect.bisect_left(detected_times_list, ot)
    candidates = []
    if pos < len(detected_times_list):
        candidates.append(detected_times_list[pos])
    if pos > 0:
        candidates.append(detected_times_list[pos - 1])
    diff_s = min((abs((c - ot) / np.timedelta64(1, "s")) for c in candidates), default=np.inf)
    if diff_s > TOL:
        missed.append({"cycle_count": cnt, "official_start": ot, "nearest_diff_s": diff_s})

missed_df = pd.DataFrame(missed)
detect_rate_2024 = 1 - len(missed_df) / len(official_2024)
print(f"\n허용오차 {TOL:.0f}초 기준 누락: {len(missed_df)}개 / {len(official_2024)}개 -- 감지율 {detect_rate_2024:.1%}")
if len(missed_df) > 0:
    print(missed_df.to_string())

공식 전체 사이클 수(2024): 94개
자체 감지한 사이클 수: 92개

허용오차 10초 기준 누락: 12개 / 94개 -- 감지율 87.2%
    cycle_count                   official_start  nearest_diff_s
0             2 2024-08-13 14:00:00.035000+00:00         344.514
1             0 2024-08-13 14:00:39.751000+00:00         304.798
2             1 2024-08-13 14:00:40.756000+00:00         303.793
3            27 2024-08-13 16:10:00.938000+00:00          10.295
4            43 2024-08-13 17:29:32.423000+00:00          22.711
5            61 2024-08-13 18:58:33.190000+00:00          18.028
6            63 2024-08-13 19:08:44.933000+00:00          13.605
7            82 2024-08-13 20:44:33.739000+00:00          22.695
8            83 2024-08-13 20:53:16.029000+00:00          22.691
9            84 2024-08-13 21:01:58.142000+00:00          22.691
10           85 2024-08-13 21:10:36.328000+00:00          22.723
11           86 2024-08-13 21:16:15.432000+00:00          15.942


## 146. PART29. 투트랙 알림 — 실시간 알림 + What-if 시뮬레이션

**🟢 쉬운 설명**
튜터님 피드백: "실시간 알림 하나, 이상이 있을 때는 다른 Slack으로 What-if 시뮬레이션 하나 —
이렇게 투트랙으로 가보자." 지금까지 우리 시스템은 "지금 무슨 일이 일어났는지"만 알려줬는데,
여기에 "만약 이랬다면 어떻게 됐을까"를 알려주는 **두 번째 채널**을 추가합니다.

- 결함이 발생하면 → "재작업했다면 vs 즉시 폐기했다면" 비용 비교 (팀 What-if 시뮬레이션:
  재작업 성공률 90% 가정 시 폐기율 51%→약 5%로 감소)
- 라인정지가 발생하면 → "이 정지 시간 동안 놓친 예상 생산량" 계산

**🔶 자세한 설명**
⚠️ **2026-07-29 반영**: What-if 메시지를 받을 별도 Slack **워크스페이스(whatif-uof4216)**가
새로 만들어졌습니다. 기존 `부품-결함-탐지-전체` 채널과는 **완전히 다른 공간**이라, 채널명만
바꾸는 걸로는 부족하고 아래 두 가지가 추가로 필요합니다:

1. 이 신규 워크스페이스에서 **Slack App(봇) 생성 + Bot Token 발급**
2. n8n에 이 토큰으로 **새 Credential**을 등록하고, `Slack 전송 (What-if)` 노드의 Credential을
   그걸로 연결(현재 `ffcell_n8n_workflow_v7_whatif워크스페이스반영.json`에는 신규
   Credential이 필요하다는 이름표만 붙여뒀고, 실제 연결은 n8n 화면에서 직접 하셔야 합니다)

채널명은 신규 워크스페이스의 기본 채널인 "일반"으로 우선 넣어뒀습니다 -- 다른 이름으로
만드셨다면 n8n 노드에서 그 이름으로 바꿔주세요.

In [ ]:
# ── PART29-1. What-if 시뮬레이션 상수 및 payload 빌더 ──────────────────
# 팀원분의 재작업 시뮬레이션 결과를 그대로 가져옴 (PLC/재작업 문서 기준)
REWORK_SUCCESS_RATE = 0.90   # 1회 재작업 시도 시 정상 복구 확률(가정)
BASELINE_SCRAP_RATE = 0.51   # 재작업 정책 없이 즉시 폐기할 경우의 시뮬레이션 폐기율
POLICY_SCRAP_RATE = 0.05     # "1회 재작업 후 실패 시 폐기" 정책 적용 시 예상 폐기율

AVG_CYCLE_TIME_S = 295.0     # 2024 데이터 기준 평균 사이클 간격(초) 근사치 -- 생산손실 추정에 사용

WHATIF_SLACK_CHANNEL_NAME = "whatif-투트랙-채널-전체"  # 2026-07-29: 신규 워크스페이스에 실제로 만든 채널
# 주의: 이건 n8n 쪽 노드 표시용 참고값일 뿐, 실제 전송 채널은 n8n Slack 노드에서 결정됩니다.
# 신규 워크스페이스라 n8n에 이 워크스페이스용 Slack Credential을 새로 등록해야 작동합니다.


def build_whatif_rework_payload(cycle_id, final_pred_kor):
    return {
        "type": "whatif_alert",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "cycle_id": int(cycle_id),
        "scenario": "rework_vs_scrap",
        "message": (
            f"[What-if] 사이클 {cycle_id} · {final_pred_kor} 결함 발생\n"
            f"재작업 없이 즉시 폐기했다면 -- 팀 시뮬레이션 기준 폐기율 {BASELINE_SCRAP_RATE:.0%}\n"
            f"1회 재작업(성공률 {REWORK_SUCCESS_RATE:.0%}) 정책 적용 시 -- 예상 폐기율 약 {POLICY_SCRAP_RATE:.0%}로 감소"
        ),
    }


def build_whatif_linestop_payload(idle_before_s, normal_median_s):
    extra_idle_s = max(0.0, idle_before_s - normal_median_s)
    lost_cycles = extra_idle_s / AVG_CYCLE_TIME_S
    return {
        "type": "whatif_alert",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "scenario": "line_stop_production_loss",
        "message": (
            f"[What-if] 라인정지로 인한 예상 생산 손실\n"
            f"정상 대비 {extra_idle_s:.0f}초 더 멈춰 있었음 -- "
            f"약 {lost_cycles:.1f}개 사이클 분량의 생산 기회 손실 추정"
        ),
    }

### PART29-2. PART27 재생 루프에 What-if 트랙 연결

In [24]:
# ── PART29-2. stream_2024_realistic를 확장 -- 결함/라인정지 시 What-if도 같이 전송 ──
def stream_2024_realistic_v2(stream_df, detected_starts, cycle_indices=None, verbose=True):
    """stream_2024_realistic(PART27)에 What-if 트랙(PART29)을 추가한 버전.
    실시간 알림(stage_alert/cycle_complete/line_stop_alert)은 그대로 보내고,
    결함·라인정지가 실제로 발생했을 때만 별도로 whatif_alert도 같이 보낸다."""
    results = []
    indices = range(len(detected_starts)) if cycle_indices is None else cycle_indices

    for i in indices:
        row = detected_starts.iloc[i]
        settle_time, idle_before_s = row["settle_time"], row["idle_before_s"]

        wait_s = min(idle_before_s, 300.0) / COMPRESSION_FACTOR
        time.sleep(wait_s)

        if idle_before_s >= LINE_STOP_THRESHOLD_2024:
            payload = build_line_stop_alert_payload(settle_time, idle_before_s, median_2024)
            if verbose:
                print(f"  {payload['message']}")
            send_to_n8n(payload)

            # What-if 트랙: 라인정지로 인한 생산손실 추정치도 별도 채널로
            whatif_payload = build_whatif_linestop_payload(idle_before_s, median_2024)
            if verbose:
                print(f"  {whatif_payload['message']}")
            send_to_n8n(whatif_payload)

        next_settle = detected_starts["settle_time"].iloc[i + 1] if i + 1 < len(detected_starts) else stream_df["_time"].max()
        cyc_window = stream_df[(stream_df["_time"] >= settle_time) & (stream_df["_time"] < next_settle)].copy()
        cyc_window["elapsed_s"] = (cyc_window["_time"] - settle_time).dt.total_seconds()

        actual_label = lookup_actual_label_2024(settle_time, cyc_2024_truth)

        if verbose:
            print(f"[자체감지 사이클 #{i + 1}] {settle_time} 부근 시작 (직전 쉬는시간 {idle_before_s:.1f}초)")

        res = stream_cycle_v3(
            cycle_id=i + 1, g=cyc_window, label=actual_label,
            known_settle_time_s=0.0, verbose=verbose,
        )

        # What-if 트랙: 결함이면 재작업 시뮬레이션도 별도 채널로
        if res["is_defect"]:
            whatif_payload = build_whatif_rework_payload(res["cycle_id"], res["final_pred_kor"])
            if verbose:
                print(f"  {whatif_payload['message']}")
            send_to_n8n(whatif_payload)

        results.append(res)

    return results


# 표본 5개(정상 3 + 결함 2, PART27과 동일 구성) -- What-if 메시지까지 같이 확인
sample_results_v2 = stream_2024_realistic_v2(stream_2024, detected_2024, cycle_indices=sample_indices)
acc_v2 = sum(r["correct"] for r in sample_results_v2) / len(sample_results_v2)
print(f"\n표본 {len(sample_results_v2)}개 기준 정확도: {acc_v2:.3f}")

[자체감지 사이클 #1] 2024-08-13 14:05:44.549000+00:00 부근 시작 (직전 쉬는시간 282.6초)
  [폴링 감지] t=60.0s -- Body1 신호 확인됨
  [폴링 감지] t=73.5s -- Body2 신호 확인됨
  [폴링 감지] t=85.5s -- Nose 신호 확인됨
  -> n8n 응답: 200
[자체감지 사이클 #25] 2024-08-13 16:05:07.328000+00:00 부근 시작 (직전 쉬는시간 280.0초)
  [폴링 감지] t=60.0s -- Body1 신호 확인됨
  [폴링 감지] t=73.5s -- Body2 신호 확인됨
  [폴링 감지] t=85.5s -- Nose 신호 확인됨
  -> n8n 응답: 200
  [라인정지 의심] 2024-08-13 17:29:55.134000+00:00 직전 291초 동안 R01 움직임이 없었습니다 (정상 중앙값 281초 대비 1.0배) -- 불량이 아니라 라인 자체가 멈췄을 가능성이 있습니다.
  -> n8n 응답: 200
  [What-if] 라인정지로 인한 예상 생산 손실
정상 대비 11초 더 멈춰 있었음 -- 약 0.0개 사이클 분량의 생산 기회 손실 추정
  -> n8n 응답: 200
[자체감지 사이클 #42] 2024-08-13 17:29:55.134000+00:00 부근 시작 (직전 쉬는시간 291.4초)
  [조기경보] 사이클 42 — Body1 단계에서 신호 미확인 (사이클 완료 대기 없이 61.5초 시점에 즉시 발신)
  -> n8n 응답: 200
  [폴링 감지] t=69.0s -- Body2 신호 확인됨
  [조기경보] 사이클 42 — Nose 단계에서 신호 미확인 (사이클 완료 대기 없이 90.0초 시점에 즉시 발신)
  -> n8n 응답: 200
  -> n8n 응답: 200
  [What-if] 사이클 42 · 노즈 미조립 결함 발생
재작업 없이 즉시 폐기했다면 -- 팀 시뮬레이션 기준 폐기율 51%
1회 재작업(성공률 90%) 정책 적용 시